# 15 · Stage 3 v5-C — overlap inference + auxiliary-head fusion

이 notebook은 **재학습하지 않는다**. v5-B `best.pt`를 고정하고 추론 방식을 검증한다.

검증 순서:

1. comma2k19 validation에서 stop / accel / decel / turn / cruise가 섞인
   **완전한 segment 25개**를 deterministic하게 선택한다.
2. `T=32`에서 stride `32 / 16 / 8`을 비교한다.
3. overlap이 있는 경우 uniform average와 center-weighted overlap-add를 비교한다.
4. overlap 방법은 tune segment에서 고르되, untouched segment holdout에서
   non-overlap보다 나빠지면 reject한다.
5. 선택된 overlap feature에 STOP / accel ordinal / steering direction /
   yaw-turn auxiliary head를 작은 grid로 fusion한다.
6. fusion weight는 tune에서만 선택하고 holdout에서 다시 검증한다.
7. 마지막으로 released 50 labels는 **선택에 사용하지 않고** target-domain
   sanity check만 한다.

최종 출력 `v5c_selection.json`은 다음 submission builder가 그대로 사용한다.


In [1]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

from google.colab import drive

try:
    drive.mount("/content/drive", force_remount=False)
except Exception:
    drive.mount("/content/drive", force_remount=True)

REPO = Path("/content/Blackbox-Detection")
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage3-sangchun"

if not (REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH,
         "--single-branch", REPO_URL, str(REPO)],
        check=True,
    )
else:
    current = subprocess.run(
        ["git", "-C", str(REPO), "branch", "--show-current"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if current != BRANCH:
        subprocess.run(
            ["git", "-C", str(REPO), "checkout", BRANCH],
            check=True,
        )
    dirty = subprocess.run(
        ["git", "-C", str(REPO), "status", "--porcelain"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if not dirty:
        subprocess.run(
            ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
            check=True,
        )
    else:
        print("WARNING: local repo dirty; git pull skipped")

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        "timm==1.0.15",
        "fvcore==0.1.5.post20221221",
        "iopath==0.1.10",
        "yacs==0.1.8",
        "einops==0.8.1",
        "easydict==1.13",
    ],
    check=True,
)

SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import torch
import yaml

from blackbox_detection.stage3.dacon_inference import infer_public_frame_mapping
from blackbox_detection.stage3.metrics import dacon_stage3_metrics
from blackbox_detection.stage3.proxy_metrics import assert_dacon_metric_contract
from blackbox_detection.stage3.schema import read_frame_table
from blackbox_detection.stage3.v5c_inference import (
    FusionConfig,
    extract_video_features_v5c,
    grid_search_fusion,
    score_proxy_table,
    score_public_labels,
)
from blackbox_detection.utils import seed_everything
from blackbox_detection.utils.checkpoint import load_checkpoint

DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATA_ROOT = DRIVE_ROOT / "DATASET"
COMMA_ROOT = DATA_ROOT / "comma2k19" / "processed" / "v1"
MANIFEST_ROOT = DRIVE_ROOT / "manifests/stage3/v1"
OUTPUT_ROOT = DRIVE_ROOT / "outputs/stage3"
PRETRAINED_ROOT = DRIVE_ROOT / "pretrained"
LOCAL_PRETRAINED_ROOT = Path("/content/pretrained")
LOCAL_PRETRAINED_ROOT.mkdir(parents=True, exist_ok=True)

CFG_PATH = REPO / "configs/stage3/vjepa21b_can_v5c.yaml"
cfg = yaml.safe_load(CFG_PATH.read_text(encoding="utf-8"))
V5B_CFG_PATH = REPO / cfg["experiment"]["source_config"]
v5b_cfg = yaml.safe_load(V5B_CFG_PATH.read_text(encoding="utf-8"))

RUN_NAME = cfg["experiment"]["name"]
RUN_DIR = OUTPUT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

stats = json.loads(
    (MANIFEST_ROOT / "target_stats.json").read_text(encoding="utf-8")
)

SEED = int(cfg["seed"])
seed_everything(SEED, deterministic=False)
assert_dacon_metric_contract()

print("GPU          :", torch.cuda.get_device_name(0))
print("v5-C config  :", CFG_PATH)
print("v5-B config  :", V5B_CFG_PATH)
print("output       :", RUN_DIR)
print("metric       : PASS")


Mounted at /content/drive
GPU          : NVIDIA L4
v5-C config  : /content/Blackbox-Detection/configs/stage3/vjepa21b_can_v5c.yaml
v5-B config  : /content/Blackbox-Detection/configs/stage3/vjepa21b_can_v5b.yaml
output       : /content/drive/MyDrive/Blackbox-Detection/outputs/stage3/vjepa21b_can_v5c_overlap_auxfusion
metric       : PASS


## 1. Load v5-B best checkpoint as a frozen inference model


In [2]:
def is_usable(path: Path, min_bytes: int = 1) -> bool:
    try:
        return path.is_file() and path.stat().st_size >= min_bytes
    except OSError:
        return False

def copy_to_local(source: Path, dest: Path, min_bytes: int = 1):
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_name(dest.name + ".tmp")
    tmp.unlink(missing_ok=True)
    with source.open("rb") as src, tmp.open("wb") as dst:
        shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
    if tmp.stat().st_size < min_bytes:
        raise OSError(f"staged file too small: {tmp.stat().st_size}")
    os.replace(tmp, dest)

VJEPA_REPO = Path("/content/vjepa2")
VJEPA_COMMIT = "45d025f636dfc58fc2426905fc4a1ab755b1c3e5"
if not (VJEPA_REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "-q",
         "https://github.com/facebookresearch/vjepa2.git",
         str(VJEPA_REPO)],
        check=True,
    )
subprocess.run(
    ["git", "-C", str(VJEPA_REPO), "fetch", "--all", "--tags"],
    check=True,
)
subprocess.run(
    ["git", "-C", str(VJEPA_REPO), "checkout", "-q", VJEPA_COMMIT],
    check=True,
)

VJEPA_NAME = "vjepa2_1_vitb_dist_vitG_384.pt"
VJEPA_DRIVE = PRETRAINED_ROOT / VJEPA_NAME
VJEPA_LOCAL = LOCAL_PRETRAINED_ROOT / VJEPA_NAME
if not is_usable(VJEPA_LOCAL, 1_000_000_000):
    if not is_usable(VJEPA_DRIVE, 1_000_000_000):
        raise FileNotFoundError(VJEPA_DRIVE)
    copy_to_local(VJEPA_DRIVE, VJEPA_LOCAL, 1_000_000_000)

SOURCE_RUN = cfg["experiment"]["source_run"]
SOURCE_CKPT_DRIVE = (
    OUTPUT_ROOT / SOURCE_RUN / cfg["experiment"]["source_checkpoint"]
)
SOURCE_CKPT_LOCAL = (
    LOCAL_PRETRAINED_ROOT
    / f"{SOURCE_RUN}__{cfg['experiment']['source_checkpoint']}"
)
if not is_usable(SOURCE_CKPT_LOCAL, 1_000_000):
    if not is_usable(SOURCE_CKPT_DRIVE, 1_000_000):
        raise FileNotFoundError(SOURCE_CKPT_DRIVE)
    copy_to_local(SOURCE_CKPT_DRIVE, SOURCE_CKPT_LOCAL, 1_000_000)

from blackbox_detection.stage3.vjepa21 import load_vjepa21_base_encoder
from blackbox_detection.stage3.v5_models import VJEPA21DenseCANV5

dc = v5b_cfg["data"]
mc = v5b_cfg["model"]
fusion_model_cfg = dict(mc["accel_fusion"])
spatial_cfg = dict(mc["spatial_pool"])

backbone = load_vjepa21_base_encoder(
    VJEPA_REPO,
    VJEPA_LOCAL,
    num_frames=int(dc["clip_len"]),
    out_layers=tuple(mc["out_layers"]),
    freeze=True,
)

model = VJEPA21DenseCANV5(
    backbone,
    freeze_backbone=True,
    feature_dim=int(mc["feature_dim"]),
    temporal_hidden=int(mc["temporal_hidden"]),
    temporal_layers=int(mc["temporal_layers"]),
    spatial_grid=tuple(spatial_cfg["grid"]),
    spatial_gate_init=float(spatial_cfg["gate_init"]),
    accel_ordinal_thresholds_mps2=mc["accel_ordinal_thresholds_mps2"],
    accel_fusion_enabled=bool(fusion_model_cfg["enabled"]),
    accel_fusion_hidden=int(fusion_model_cfg["hidden"]),
    accel_fusion_gate_init=float(fusion_model_cfg["gate_init"]),
    accel_fusion_detach_ordinal_inputs=bool(
        fusion_model_cfg["detach_ordinal_inputs"]
    ),
    stop_thresholds_mps=mc["stop_thresholds_mps"],
    turn_yaw_thresholds_rps=mc["turn_yaw_thresholds_rps"],
    steer_activity_thresholds=mc["steer_activity_thresholds"],
    brake_thresholds_bar=mc["brake_thresholds_bar"],
    throttle_thresholds_pct=mc["throttle_thresholds_pct"],
)

meta = load_checkpoint(
    SOURCE_CKPT_LOCAL,
    model=model,
    optimizer=None,
    scheduler=None,
    map_location="cpu",
    strict=True,
    restore_rng_state=False,
)

device = torch.device("cuda")
model.to(device).eval()
model.requires_grad_(False)

STOP_THRESH = [float(x) for x in mc["stop_thresholds_mps"]]
ACCEL_THRESH = [float(x) for x in mc["accel_ordinal_thresholds_mps2"]]
TURN_THRESH = [float(x) for x in mc["turn_yaw_thresholds_rps"]]

print("checkpoint :", SOURCE_CKPT_LOCAL)
print("epoch      :", meta.get("epoch"))
print("T          :", dc["clip_len"])
print("stop thr   :", STOP_THRESH)
print("accel thr  :", ACCEL_THRESH)
print("turn thr   :", TURN_THRESH)


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


checkpoint : /content/pretrained/vjepa21b_can_v5b_last2_ft__best.pt
epoch      : 2
T          : 32
stop thr   : [0.1, 0.3, 0.5, 1.0, 2.0]
accel thr  : [0.1, 0.2, 0.3, 0.5]
turn thr   : [0.01, 0.03, 0.05]


## 2. Build a diverse complete-segment validation subset

앞 800 window를 다시 쓰지 않는다. metadata만 훑어서 stop / hard decel /
hard accel / turn / cruise가 많은 segment를 각 5개씩 골라 **완전한 segment**
단위로 평가한다. 같은 segment의 일부 window가 tune과 holdout으로 갈라지지
않는다.


In [3]:
val_manifest = pd.read_csv(MANIFEST_ROOT / "comma_val_id.csv")

def segment_profile(row):
    meta = read_frame_table(COMMA_ROOT / row.metadata_relpath)

    speed = meta["speed_mps"].to_numpy(dtype=np.float64)
    accel = meta["accel_from_speed_mps2"].to_numpy(dtype=np.float64)
    yaw = meta["yaw_rate_rps"].to_numpy(dtype=np.float64)

    vs = meta["valid_speed"].to_numpy(dtype=bool) & np.isfinite(speed)
    va = (
        meta["valid_accel_from_speed"].to_numpy(dtype=bool)
        & np.isfinite(accel)
    )
    vy = meta["valid_yaw"].to_numpy(dtype=bool) & np.isfinite(yaw)

    stop_fraction = float(np.mean(speed[vs] <= 1.0)) if vs.any() else 0.0
    hard_accel_fraction = (
        float(np.mean(accel[va] > 0.30)) if va.any() else 0.0
    )
    hard_decel_fraction = (
        float(np.mean(accel[va] < -0.30)) if va.any() else 0.0
    )
    turn_fraction = (
        float(np.mean(np.abs(yaw[vy]) > 0.03)) if vy.any() else 0.0
    )
    cruise_fraction = 0.0
    if vs.any() and va.any() and vy.any() and len(speed) == len(accel) == len(yaw):
        common = vs & va & vy
        if common.any():
            cruise_fraction = float(
                np.mean(
                    (speed[common] > 3.0)
                    & (np.abs(accel[common]) < 0.10)
                    & (np.abs(yaw[common]) < 0.01)
                )
            )

    return {
        "route_id": str(row.route_id),
        "segment_id": str(row.segment_id),
        "segment_key": f"{row.route_id}/{row.segment_id}",
        "num_frames": int(row.num_frames),
        "video_relpath": str(row.video_relpath),
        "metadata_relpath": str(row.metadata_relpath),
        "stop_start": stop_fraction,
        "hard_decel": hard_decel_fraction,
        "hard_accel": hard_accel_fraction,
        "turn": turn_fraction,
        "cruise": cruise_fraction,
    }

profiles = pd.DataFrame(
    segment_profile(row)
    for row in val_manifest.itertuples(index=False)
)

vc = cfg["validation"]
buckets = ["stop_start", "hard_decel", "hard_accel", "turn", "cruise"]
per_bucket = int(vc["per_bucket"])
num_segments = int(vc["num_segments"])

selected_rows = []
used = set()
for bucket in buckets:
    ranked = profiles.sort_values(
        [bucket, "segment_key"],
        ascending=[False, True],
    )
    taken = 0
    for _, row in ranked.iterrows():
        key = row["segment_key"]
        if key in used:
            continue
        item = row.to_dict()
        item["selection_bucket"] = bucket
        selected_rows.append(item)
        used.add(key)
        taken += 1
        if taken >= per_bucket:
            break

# Fill if bucket overlap prevented the requested total.
if len(selected_rows) < num_segments:
    remaining = profiles[~profiles["segment_key"].isin(used)].sort_values(
        "segment_key"
    )
    fill_n = num_segments - len(selected_rows)
    if fill_n > 0 and len(remaining):
        positions = np.linspace(
            0, len(remaining) - 1,
            num=min(fill_n, len(remaining)),
            dtype=int,
        )
        for pos in np.unique(positions):
            item = remaining.iloc[int(pos)].to_dict()
            item["selection_bucket"] = "coverage"
            selected_rows.append(item)

selected = pd.DataFrame(selected_rows).head(num_segments).copy()

# Stratified-by-selection-bucket holdout: roughly one of every 5 segments.
selected["split"] = "tune"
for bucket, idx in selected.groupby("selection_bucket", sort=False).groups.items():
    ids = list(idx)
    # Each main bucket contributes 5; reserve the last one.
    selected.loc[ids[-1], "split"] = "holdout"

# Guarantee at least 4 holdout segments.
if int((selected["split"] == "holdout").sum()) < 4:
    selected.loc[selected.index[::5], "split"] = "holdout"

selected_path = RUN_DIR / "selected_segments.csv"
selected.to_csv(selected_path, index=False)

print("selected:", len(selected))
print(selected["selection_bucket"].value_counts().to_dict())
print(selected["split"].value_counts().to_dict())
display(
    selected[
        [
            "segment_key", "selection_bucket", "split",
            "stop_start", "hard_decel", "hard_accel", "turn", "cruise"
        ]
    ]
)


selected: 25
{'stop_start': 5, 'hard_decel': 5, 'hard_accel': 5, 'turn': 5, 'cruise': 5}
{'tune': 20, 'holdout': 5}


,segment_key,selection_bucket,split,stop_start,hard_decel,hard_accel,turn,cruise
0,99c94dc769b5d96e|2018-09-19--16-09-16/10,stop_start,tune,1.000000,0.000000,0.000000,0.000000,0.000000
1,b0c9d2329ad1606b|2018-11-11--13-08-44/13,stop_start,tune,1.000000,0.000000,0.000000,0.000000,0.000000
2,99c94dc769b5d96e|2018-09-19--16-09-16/35,stop_start,tune,0.955000,0.060100,0.000000,0.000000,0.000000
3,99c94dc769b5d96e|2018-08-03--14-04-02/26,stop_start,tune,0.929883,0.071906,0.000000,0.000000,0.000000
4,99c94dc769b5d96e|2018-07-20--16-07-46/9,stop_start,holdout,0.911519,0.102007,0.000000,0.000000,0.000000
5,99c94dc769b5d96e|2018-07-06--11-17-49/5,hard_decel,tune,0.000000,0.585284,0.277592,0.001669,0.026756
6,99c94dc769b5d96e|2018-05-18--16-00-42/67,hard_decel,tune,0.000000,0.566890,0.071906,0.015025,0.086957
7,99c94dc769b5d96e|2018-06-15--19-55-32/7,hard_decel,tune,0.058431,0.558528,0.267559,0.026711,0.000000
8,99c94dc769b5d96e|2018-09-19--16-09-16/31,hard_decel,tune,0.000000,0.476589,0.107023,0.016694,0.053512
9,99c94dc769b5d96e|2018-06-15--19-55-32/32,hard_decel,holdout,0.000000,0.473934,0.066351,0.301887,0.000000


## 3. Run overlap variants and cache dense features

각 stride의 model forward는 한 번만 수행하고, 같은 window prediction으로
`center_floor=1.0`과 `0.25`를 동시에 aggregate한다. 따라서 두 weighting
방식 비교 때문에 GPU 추론을 두 번 하지 않는다.

Drive cache가 있으면 런타임이 끊겨도 다시 추론하지 않는다.


In [4]:
FEATURE_DIR = RUN_DIR / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

def attach_truth(pred, row):
    meta = read_frame_table(COMMA_ROOT / row.metadata_relpath).reset_index(drop=True)
    if len(meta) != len(pred):
        raise RuntimeError(
            f"frame count mismatch {row.segment_key}: "
            f"video={len(pred)}, metadata={len(meta)}"
        )
    out = pred.copy()
    out.insert(0, "segment_key", str(row.segment_key))
    out.insert(1, "route_id", str(row.route_id))
    out.insert(2, "segment_id", str(row.segment_id))
    out["gt_speed_mps"] = meta["speed_mps"].to_numpy(dtype=np.float32)
    out["gt_accel_mps2"] = meta[
        "accel_from_speed_mps2"
    ].to_numpy(dtype=np.float32)
    out["gt_steering_deg"] = meta["steering_deg"].to_numpy(dtype=np.float32)
    out["valid_speed"] = meta["valid_speed"].to_numpy(dtype=bool)
    out["valid_accel"] = meta[
        "valid_accel_from_speed"
    ].to_numpy(dtype=bool)
    out["valid_steer"] = meta["valid_steer"].to_numpy(dtype=bool)
    return out

strides = [int(x) for x in vc["strides"]]
floors = [float(x) for x in vc["center_floors"]]
method_tables = {}
runtime_rows = []

for stride in strides:
    cache_paths = {
        floor: FEATURE_DIR / f"features_s{stride}_c{floor:.2f}.csv"
        for floor in floors
    }
    if all(path.is_file() for path in cache_paths.values()):
        print(f"stride={stride}: CACHE HIT")
        for floor, path in cache_paths.items():
            method_tables[(stride, floor)] = pd.read_csv(path)
        continue

    per_floor_parts = {floor: [] for floor in floors}
    t0 = time.perf_counter()
    window_count = 0

    for i, row in enumerate(selected.itertuples(index=False), start=1):
        video_path = COMMA_ROOT / row.video_relpath
        variants = extract_video_features_v5c(
            model,
            video_path,
            target_stats=stats,
            stop_thresholds_mps=STOP_THRESH,
            accel_thresholds_mps2=ACCEL_THRESH,
            turn_thresholds_rps=TURN_THRESH,
            input_height=int(cfg["data"]["input_height"]),
            input_width=int(cfg["data"]["input_width"]),
            raw_stride=1,
            raw_offset=0,
            clip_len=int(cfg["data"]["clip_len"]),
            window_stride=stride,
            batch_size=int(cfg["data"]["batch_size"]),
            center_floors=floors,
            device=device,
            use_amp=bool(vc["use_amp"]),
        )
        n = len(next(iter(variants.values())))
        if n <= int(cfg["data"]["clip_len"]):
            windows = 1
        else:
            windows = (
                math.ceil(
                    (n - int(cfg["data"]["clip_len"])) / stride
                )
                + 1
            )
        window_count += windows

        for floor, table in variants.items():
            per_floor_parts[floor].append(attach_truth(table, row))

        if i == 1 or i % 5 == 0 or i == len(selected):
            print(f"stride={stride}: {i}/{len(selected)} segments")

    elapsed = time.perf_counter() - t0
    total_frames = int(selected["num_frames"].sum())
    runtime_rows.append(
        {
            "stride": stride,
            "seconds": elapsed,
            "frames": total_frames,
            "windows_approx": window_count,
            "seconds_per_600_frames": elapsed / total_frames * 600.0,
        }
    )

    for floor in floors:
        table = pd.concat(per_floor_parts[floor], ignore_index=True)
        table.to_csv(cache_paths[floor], index=False)
        method_tables[(stride, floor)] = table
        print("saved:", cache_paths[floor])

runtime_df = pd.DataFrame(runtime_rows)
display(runtime_df)


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


stride=32: 1/25 segments


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context m

stride=32: 5/25 segments


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context m

...

/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context m

stride=8: 20/25 segments


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context m

stride=8: 25/25 segments
saved: /content/drive/MyDrive/Blackbox-Detection/outputs/stage3/vjepa21b_can_v5c_overlap_auxfusion/features/features_s8_c1.00.csv
saved: /content/drive/MyDrive/Blackbox-Detection/outputs/stage3/vjepa21b_can_v5c_overlap_auxfusion/features/features_s8_c0.25.csv


,stride,seconds,frames,windows_approx,seconds_per_600_frames
0,32,266.688577,14617,463,10.947058
1,16,436.848440,14617,901,17.931796
2,8,837.548010,14617,1756,34.379750


## 4. Compare overlap-only methods on tune / holdout / full


In [5]:
proxy_rules = vc["proxy_rules"]
zero_fusion = FusionConfig()

def subset_for(table, split):
    keys = set(
        selected.loc[selected["split"] == split, "segment_key"].astype(str)
    )
    return table[table["segment_key"].astype(str).isin(keys)].reset_index(drop=True)

method_rows = []
method_scores = {}

for (stride, floor), table in method_tables.items():
    row = {
        "method": f"s{stride}_c{floor:.2f}",
        "stride": stride,
        "center_floor": floor,
    }
    for split_name in ("tune", "holdout"):
        part = subset_for(table, split_name)
        score = score_proxy_table(
            part,
            proxy_rules,
            fusion=zero_fusion,
            stop_thresholds_mps=STOP_THRESH,
            accel_thresholds_mps2=ACCEL_THRESH,
            turn_thresholds_rps=TURN_THRESH,
        )
        method_scores[(stride, floor, split_name)] = score
        row[f"{split_name}_stage3"] = score[
            "proxy/robust_mean_stage3_score"
        ]
        row[f"{split_name}_accel"] = score[
            "proxy/robust_mean_accel_macro_f1"
        ]
        row[f"{split_name}_steer"] = score[
            "proxy/robust_mean_steer_macro_f1"
        ]

    full = score_proxy_table(
        table,
        proxy_rules,
        fusion=zero_fusion,
        stop_thresholds_mps=STOP_THRESH,
        accel_thresholds_mps2=ACCEL_THRESH,
        turn_thresholds_rps=TURN_THRESH,
    )
    method_scores[(stride, floor, "full")] = full
    row["full_stage3"] = full["proxy/robust_mean_stage3_score"]
    row["accel_corr"] = full["diag/accel/correlation"]
    row["accel_std_ratio"] = full[
        "diag/accel/pred_to_gt_std_ratio"
    ]
    row["accel_mae"] = full["diag/accel/mae"]
    method_rows.append(row)

method_df = pd.DataFrame(method_rows).sort_values(
    ["tune_stage3", "stride"],
    ascending=[False, False],
).reset_index(drop=True)

display(method_df)

baseline_key = (32, 1.0)
if baseline_key not in method_tables:
    raise RuntimeError("required baseline s32_c1.00 is missing")

best_tune = float(method_df["tune_stage3"].max())
tol = float(vc["overlap_tie_tolerance"])
eligible = method_df[
    method_df["tune_stage3"] >= best_tune - tol
].copy()

# Within statistical near-ties, prefer cheaper inference.
eligible = eligible.sort_values(
    ["stride", "tune_stage3"],
    ascending=[False, False],
)
chosen_overlap = eligible.iloc[0]

chosen_key = (
    int(chosen_overlap["stride"]),
    float(chosen_overlap["center_floor"]),
)

base_hold = method_scores[(*baseline_key, "holdout")][
    "proxy/robust_mean_stage3_score"
]
chosen_hold = method_scores[(*chosen_key, "holdout")][
    "proxy/robust_mean_stage3_score"
]

if chosen_hold < base_hold - float(vc["overlap_holdout_max_drop"]):
    print(
        "OVERLAP REJECTED by holdout:",
        chosen_key,
        "holdout", chosen_hold,
        "baseline", base_hold,
    )
    chosen_key = baseline_key
else:
    print("OVERLAP ACCEPTED:", chosen_key)

selected_features = method_tables[chosen_key]
print("selected overlap:", chosen_key)


,method,stride,center_floor,tune_stage3,tune_accel,tune_steer,holdout_stage3,holdout_accel,holdout_steer,full_stage3,accel_corr,accel_std_ratio,accel_mae
0,s8_c0.25,8,0.25,0.698960,0.718756,0.652767,0.632182,0.676051,0.529819,0.688204,0.712745,0.680221,0.217117
1,s8_c1.00,8,1.00,0.697667,0.717142,0.652227,0.631304,0.676243,0.526447,0.686933,0.713473,0.655446,0.217830
2,s16_c0.25,16,0.25,0.695935,0.715047,0.651342,0.629983,0.672446,0.530903,0.685450,0.703291,0.677513,0.220761
3,s16_c1.00,16,1.00,0.693606,0.712149,0.650341,0.626055,0.668396,0.527259,0.682880,0.699730,0.656591,0.222755
4,s32_c0.25,32,0.25,0.691513,0.708840,0.651084,0.598525,0.627191,0.531638,0.677546,0.676344,0.669048,0.229139
5,s32_c1.00,32,1.00,0.691446,0.708775,0.651013,0.598933,0.627774,0.531638,0.677570,0.676311,0.668995,0.229148


OVERLAP ACCEPTED: (8, 0.25)
selected overlap: (8, 0.25)


## 5. Tune auxiliary fusion on tune segments only

192개의 작은 grid를 돈다. GPU inference는 다시 하지 않는다.

- STOP ordinal → speed boundary 주변 log-odds
- accel ordinal → ACCEL/DECEL boundary 주변 log-odds
- steering direction → continuous steering LEFT/RIGHT margin
- yaw-turn ordinal → 방향 보조 evidence

모든 weight가 0이면 continuous-only 결정과 **정확히 동일**하다.


In [6]:
fc = cfg["fusion_search"]
tune_features = subset_for(selected_features, "tune")
hold_features = subset_for(selected_features, "holdout")

fusion_grid = grid_search_fusion(
    tune_features,
    proxy_rules,
    stop_thresholds_mps=STOP_THRESH,
    accel_thresholds_mps2=ACCEL_THRESH,
    turn_thresholds_rps=TURN_THRESH,
    stop_weights=fc["stop_weights"],
    accel_weights=fc["accel_weights"],
    steer_weights=fc["steer_weights"],
    turn_weights=fc["turn_weights"],
    stop_temperature_mps=fc["stop_temperature_mps"],
    accel_temperature_mps2=fc["accel_temperature_mps2"],
    steer_temperature_deg=fc["steer_temperature_deg"],
)

display(fusion_grid.head(20))

best_row = fusion_grid.iloc[0]
candidate = FusionConfig(
    stop_weight=float(best_row["stop_weight"]),
    accel_weight=float(best_row["accel_weight"]),
    steer_weight=float(best_row["steer_weight"]),
    turn_weight=float(best_row["turn_weight"]),
    stop_temperature_mps=float(best_row["stop_temperature_mps"]),
    accel_temperature_mps2=float(best_row["accel_temperature_mps2"]),
    steer_temperature_deg=float(best_row["steer_temperature_deg"]),
)

def scored(table, fusion):
    return score_proxy_table(
        table,
        proxy_rules,
        fusion=fusion,
        stop_thresholds_mps=STOP_THRESH,
        accel_thresholds_mps2=ACCEL_THRESH,
        turn_thresholds_rps=TURN_THRESH,
    )

base_tune = scored(tune_features, zero_fusion)
base_hold = scored(hold_features, zero_fusion)
base_full = scored(selected_features, zero_fusion)

cand_tune = scored(tune_features, candidate)
cand_hold = scored(hold_features, candidate)
cand_full = scored(selected_features, candidate)

delta_tune = (
    cand_tune["proxy/robust_mean_stage3_score"]
    - base_tune["proxy/robust_mean_stage3_score"]
)
delta_hold = (
    cand_hold["proxy/robust_mean_stage3_score"]
    - base_hold["proxy/robust_mean_stage3_score"]
)
delta_full = (
    cand_full["proxy/robust_mean_stage3_score"]
    - base_full["proxy/robust_mean_stage3_score"]
)

hold_rule_deltas = {}
for name in proxy_rules:
    key = f"proxy/{name}/stage3_score"
    hold_rule_deltas[name] = cand_hold[key] - base_hold[key]

worst_hold_rule_delta = min(hold_rule_deltas.values())

accepted_fusion = (
    delta_hold >= float(fc["min_holdout_delta"])
    and delta_full >= float(fc["min_full_delta"])
    and worst_hold_rule_delta
        >= -float(fc["max_single_rule_holdout_drop"])
)

production_fusion = candidate if accepted_fusion else zero_fusion

comparison = pd.DataFrame(
    [
        {
            "variant": "overlap_only",
            "tune": base_tune["proxy/robust_mean_stage3_score"],
            "holdout": base_hold["proxy/robust_mean_stage3_score"],
            "full": base_full["proxy/robust_mean_stage3_score"],
            "accel": base_full["proxy/robust_mean_accel_macro_f1"],
            "steer": base_full["proxy/robust_mean_steer_macro_f1"],
        },
        {
            "variant": "candidate_aux_fusion",
            "tune": cand_tune["proxy/robust_mean_stage3_score"],
            "holdout": cand_hold["proxy/robust_mean_stage3_score"],
            "full": cand_full["proxy/robust_mean_stage3_score"],
            "accel": cand_full["proxy/robust_mean_accel_macro_f1"],
            "steer": cand_full["proxy/robust_mean_steer_macro_f1"],
        },
    ]
)
display(comparison)

print("candidate fusion:", candidate.as_dict())
print("holdout rule deltas:", hold_rule_deltas)
print("fusion accepted:", accepted_fusion)
print("production fusion:", production_fusion.as_dict())


,stop_weight,accel_weight,steer_weight,turn_weight,stop_temperature_mps,accel_temperature_mps2,steer_temperature_deg,robust_stage3,robust_min_stage3,robust_accel,robust_steer
0,0.75,0.00,0.30,0.30,0.25,0.1,2.0,0.712631,0.696848,0.731080,0.669583
1,0.75,0.00,0.00,0.30,0.25,0.1,2.0,0.712618,0.697651,0.731080,0.669539
2,0.75,0.00,0.50,0.30,0.25,0.1,2.0,0.712561,0.694842,0.731080,0.669351
3,0.75,0.00,0.15,0.30,0.25,0.1,2.0,0.712504,0.697020,0.731080,0.669160
4,0.75,0.15,0.30,0.30,0.25,0.1,2.0,0.712291,0.698047,0.730595,0.669583
5,0.75,0.15,0.00,0.30,0.25,0.1,2.0,0.712278,0.698850,0.730595,0.669539
6,0.75,0.15,0.50,0.30,0.25,0.1,2.0,0.712221,0.696041,0.730595,0.669351
7,0.75,0.15,0.15,0.30,0.25,0.1,2.0,0.712164,0.698219,0.730595,0.669160
8,0.50,0.00,0.30,0.30,0.25,0.1,2.0,0.711941,0.695716,0.730094,0.669583
9,0.50,0.00,0.00,0.30,0.25,0.1,2.0,0.711927,0.696519,0.730094,0.669539


,variant,tune,holdout,full,accel,steer
0,overlap_only,0.698960,0.632182,0.688204,0.713302,0.629642
1,candidate_aux_fusion,0.712631,0.649522,0.702905,0.725738,0.649626


candidate fusion: {'stop_weight': 0.75, 'accel_weight': 0.0, 'steer_weight': 0.3, 'turn_weight': 0.3, 'stop_temperature_mps': 0.25, 'accel_temperature_mps2': 0.1, 'steer_temperature_deg': 2.0}
holdout rule deltas: {'sensitive': 0.042600876062242055, 'medium': 0.009511729054296092, 'conservative': -9.060157215823317e-05}
fusion accepted: True
production fusion: {'stop_weight': 0.75, 'accel_weight': 0.0, 'steer_weight': 0.3, 'turn_weight': 0.3, 'stop_temperature_mps': 0.25, 'accel_temperature_mps2': 0.1, 'steer_temperature_deg': 2.0}


## 6. Save v5-C selection


In [7]:
selection_payload = {
    "version": 1,
    "source_run": SOURCE_RUN,
    "source_checkpoint": cfg["experiment"]["source_checkpoint"],
    "clip_len": int(cfg["data"]["clip_len"]),
    "overlap": {
        "stride": int(chosen_key[0]),
        "center_floor": float(chosen_key[1]),
    },
    "fusion": production_fusion.as_dict(),
    "fusion_candidate": candidate.as_dict(),
    "fusion_accepted": bool(accepted_fusion),
    "validation": {
        "num_segments": int(len(selected)),
        "tune_segments": int((selected["split"] == "tune").sum()),
        "holdout_segments": int((selected["split"] == "holdout").sum()),
        "overlap_only_tune": float(
            base_tune["proxy/robust_mean_stage3_score"]
        ),
        "overlap_only_holdout": float(
            base_hold["proxy/robust_mean_stage3_score"]
        ),
        "overlap_only_full": float(
            base_full["proxy/robust_mean_stage3_score"]
        ),
        "candidate_tune": float(
            cand_tune["proxy/robust_mean_stage3_score"]
        ),
        "candidate_holdout": float(
            cand_hold["proxy/robust_mean_stage3_score"]
        ),
        "candidate_full": float(
            cand_full["proxy/robust_mean_stage3_score"]
        ),
        "candidate_delta_tune": float(delta_tune),
        "candidate_delta_holdout": float(delta_hold),
        "candidate_delta_full": float(delta_full),
        "candidate_holdout_rule_deltas": {
            k: float(v) for k, v in hold_rule_deltas.items()
        },
    },
    "note": (
        "Proxy thresholds are diagnostic only. Fusion weights were selected "
        "on comma2k19 tune segments and gated on untouched comma2k19 holdout; "
        "released DACON labels are not used for selection."
    ),
}

selection_path = RUN_DIR / "v5c_selection.json"
selection_path.write_text(
    json.dumps(selection_payload, indent=2),
    encoding="utf-8",
)

method_df.to_csv(RUN_DIR / "overlap_comparison.csv", index=False)
fusion_grid.to_csv(RUN_DIR / "fusion_grid.csv", index=False)

print(json.dumps(selection_payload, indent=2))
print("saved:", selection_path)


{
  "version": 1,
  "source_run": "vjepa21b_can_v5b_last2_ft",
  "source_checkpoint": "best.pt",
  "clip_len": 32,
  "overlap": {
    "stride": 8,
    "center_floor": 0.25
  },
  "fusion": {
    "stop_weight": 0.75,
    "accel_weight": 0.0,
    "steer_weight": 0.3,
    "turn_weight": 0.3,
    "stop_temperature_mps": 0.25,
    "accel_temperature_mps2": 0.1,
    "steer_temperature_deg": 2.0
  },
  "fusion_candidate": {
    "stop_weight": 0.75,
    "accel_weight": 0.0,
    "steer_weight": 0.3,
    "turn_weight": 0.3,
    "stop_temperature_mps": 0.25,
    "accel_temperature_mps2": 0.1,
    "steer_temperature_deg": 2.0
  },
  "fusion_accepted": true,
  "validation": {
    "num_segments": 25,
    "tune_segments": 20,
    "holdout_segments": 5,
    "overlap_only_tune": 0.6989596081265792,
    "overlap_only_holdout": 0.6321817390511525,
    "overlap_only_full": 0.6882044225059177,
    "candidate_tune": 0.7126309103781802,
    "candidate_holdout": 0.6495224068992792,
    "candidate_full": 0.702

## 7. Optional released-label sanity check

이 셀은 `Baseline.zip`의 공개 50 labels를 사용한다. **weight를 다시 고르지 않는다.**
이미 위에서 확정한 overlap/fusion을 그대로 적용해서 target-domain에서
catastrophic sign/threshold 문제가 없는지만 본다.


In [9]:
if bool(cfg["public_sanity"]["enabled"]):
    baseline_candidates = [
        DRIVE_ROOT / "Baseline.zip",
        Path("/content/Baseline.zip"),
    ]
    baseline_zip = next(
        (p for p in baseline_candidates if p.is_file()),
        None,
    )

    if baseline_zip is None:
        print("PUBLIC SANITY SKIP: Baseline.zip not found")
    else:
        public_root = Path("/content/v5c_public_stage3")
        if public_root.exists():
            shutil.rmtree(public_root)
        public_root.mkdir(parents=True)

        def find_stage3_root(candidates):
            for candidate in candidates:
                p = Path(candidate)
                if not p.exists():
                    continue

                label_files = (
                    [p]
                    if p.name.lower() == "labels.csv"
                    else list(p.rglob("labels.csv"))
                )

                for labels_path in label_files:
                    parent = labels_path.parent
                    has_video = (
                        any(parent.rglob("*.mp4"))
                        or any(parent.rglob("*.MP4"))
                    )
                    if has_video:
                        return parent, labels_path

            return None, None

        with zipfile.ZipFile(baseline_zip) as zf:
            members = [
                name
                for name in zf.namelist()
                if (
                    "/stage3/" in name.lower()
                    or name.lower().startswith("data/stage3/")
                    or name.lower().startswith("stage3/")
                )
            ]

            if not members:
                raise RuntimeError(
                    "Baseline.zip contains no recognizable Stage 3 members"
                )

            for name in members:
                zf.extract(name, public_root)

        stage3_root, labels_path = find_stage3_root([public_root])

        if stage3_root is None or labels_path is None:
            raise RuntimeError(
                "Could not resolve public Stage 3 root after extraction"
            )

        labels = pd.read_csv(labels_path)

        video_index = {}
        for p in sorted(
            [
                *stage3_root.rglob("*.mp4"),
                *stage3_root.rglob("*.MP4"),
            ]
        ):
            video_index.setdefault(p.stem, p)

        missing_videos = sorted(
            set(labels["ID"]) - set(video_index)
        )
        if missing_videos:
            raise FileNotFoundError(
                f"Missing public videos for IDs: {missing_videos}"
            )

        print("Baseline.zip :", baseline_zip)
        print("Stage 3 root:", stage3_root)
        print("labels      :", labels_path)
        print("rows / IDs  :", len(labels), labels["ID"].nunique())

        public_parts = []

        for video_id, subset in labels.groupby("ID", sort=True):
            video_path = video_index[str(video_id)]

            raw_stride, raw_offset = infer_public_frame_mapping(
                subset
            )

            variants = extract_video_features_v5c(
                model,
                video_path,
                target_stats=stats,
                stop_thresholds_mps=STOP_THRESH,
                accel_thresholds_mps2=ACCEL_THRESH,
                turn_thresholds_rps=TURN_THRESH,
                input_height=int(cfg["data"]["input_height"]),
                input_width=int(cfg["data"]["input_width"]),
                raw_stride=raw_stride,
                raw_offset=raw_offset,
                clip_len=int(cfg["data"]["clip_len"]),
                window_stride=int(chosen_key[0]),
                batch_size=int(cfg["data"]["batch_size"]),
                center_floors=[float(chosen_key[1])],
                device=device,
                use_amp=False,
            )

            table = variants[float(chosen_key[1])].copy()
            table.insert(0, "ID", str(video_id))
            public_parts.append(table)

            print(
                video_id,
                "stride/offset",
                raw_stride,
                raw_offset,
                "10-Hz frames",
                len(table),
            )

        public_features = pd.concat(
            public_parts,
            ignore_index=True,
        )

        cal_path = (
            REPO / cfg["public_sanity"]["calibration"]
        )
        calibration = json.loads(
            cal_path.read_text(encoding="utf-8")
        )

        public_no_aux = score_public_labels(
            public_features,
            labels,
            calibration,
            fusion=zero_fusion,
            stop_thresholds_mps=STOP_THRESH,
            accel_thresholds_mps2=ACCEL_THRESH,
            turn_thresholds_rps=TURN_THRESH,
        )

        public_selected = score_public_labels(
            public_features,
            labels,
            calibration,
            fusion=production_fusion,
            stop_thresholds_mps=STOP_THRESH,
            accel_thresholds_mps2=ACCEL_THRESH,
            turn_thresholds_rps=TURN_THRESH,
        )

        public_report = {
            "overlap_only": public_no_aux,
            "selected_v5c": public_selected,
            "delta_stage3": float(
                public_selected["stage3_score"]
                - public_no_aux["stage3_score"]
            ),
            "warning": (
                "Only 50 released labels. "
                "This is a sanity check, "
                "not a hyperparameter selection set."
            ),
        }

        (RUN_DIR / "public_sanity.json").write_text(
            json.dumps(public_report, indent=2),
            encoding="utf-8",
        )

        public_features.to_csv(
            RUN_DIR / "public_v5b_overlap_features.csv",
            index=False,
        )

        print(json.dumps(public_report, indent=2))

Baseline.zip : /content/drive/MyDrive/Blackbox-Detection/Baseline.zip
Stage 3 root: /content/v5c_public_stage3/data/stage3
labels      : /content/v5c_public_stage3/data/stage3/labels.csv
rows / IDs  : 50 5


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


OPEN_001 stride/offset 2 0 10-Hz frames 600


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


OPEN_002 stride/offset 2 0 10-Hz frames 601


...

OPEN_003 stride/offset 2 0 10-Hz frames 599


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


OPEN_004 stride/offset 2 0 10-Hz frames 599


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


OPEN_005 stride/offset 2 0 10-Hz frames 599
{
  "overlap_only": {
    "stage3_score": 0.6776217038539554,
    "accel_macro_f1": 0.7628489326765189,
    "steer_macro_f1": 0.47875816993464054,
    "steer_eval_frames": 47
  },
  "selected_v5c": {
    "stage3_score": 0.73081964969896,
    "accel_macro_f1": 0.7628489326765189,
    "steer_macro_f1": 0.656084656084656,
    "steer_eval_frames": 47
  },
  "delta_stage3": 0.05319794584500459,
  "warning": "Only 50 released labels. This is a sanity check, not a hyperparameter selection set."
}


## 다음 단계

`v5c_selection.json`, `overlap_comparison.csv`, `public_sanity.json`(있으면)을
업로드한다. 그 결과를 보고 선택을 확정한 뒤 **Stage1 A7을 byte-lock한
v5-C submission builder**를 만든다.

공개 50 labels가 나쁘다고 여기서 grid를 다시 맞추지 않는다. 그 50개는 이미
기존 calibration에 사용된 작은 표본이므로 재튜닝하면 과적합 위험이 크다.
